In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from lifelines import KaplanMeierFitter, CoxPHFitter

In [ ]:
df = pd.read_csv("../data/processed/cleaned_telco.csv")

df.head()

In [ ]:
survival_df = df[df["tenure"] > 0].copy()

survival_df[["customerID", "tenure", "Churn"]].head()

In [ ]:
kmf = KaplanMeierFitter()

kmf.fit(
    durations=survival_df["tenure"],
    event_observed=survival_df["Churn"],
    label="Customer Survival"
)

print("Kaplan-Meier model fitted successfully!")

In [ ]:
plt.figure(figsize=(10, 6))

kmf.plot_survival_function()

plt.title("Customer Survival Curve")
plt.xlabel("Tenure (Months)")
plt.ylabel("Survival Probability")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
print(kmf.predict([6, 12, 24, 36, 48, 60]))

In [ ]:
cox_df = survival_df[
    ["tenure", "Churn", "MonthlyCharges",
     "SeniorCitizen", "Contract", "InternetService"]
].copy()

cox_df = pd.get_dummies(
    cox_df,
    columns=["Contract", "InternetService"],
    drop_first=True,
    dtype=int
)

cox_df.head()

In [ ]:
from sklearn.model_selection import train_test_split

cox_train, cox_test = train_test_split(
    cox_df,
    test_size=0.2,
    random_state=42,
    stratify=cox_df["Churn"]
)

In [ ]:
cox = CoxPHFitter(penalizer=0.1)

cox.fit(
    cox_train,
    duration_col="tenure",
    event_col="Churn"
)

print("Cox model trained successfully!")

In [ ]:
cox.print_summary()

print("Test Concordance Index:",
      cox.score(
          cox_test,
          scoring_method="concordance_index"
      ))

In [ ]:
customer = cox_test.drop(
    columns=["tenure", "Churn"]
).iloc[[0]]

cox.predict_survival_function(customer).plot()

plt.title("Individual Customer Survival Probability")
plt.xlabel("Tenure (Months)")
plt.ylabel("Survival Probability")
plt.show()

In [ ]:
joblib.dump(cox, "../models/cox_model.pkl")

joblib.dump(kmf, "../models/kaplan_meier.pkl")

print("Survival models saved successfully!")